In [25]:
# 1. Definirea parametrilor (Sectiunea 3.2 din articol)
nbits = 256 # Pentru RSA-512 (p si q au 256 biti fiecare)

# Generam numerele prime p si q
p = next_prime(2^nbits + randint(1, 10^6))
q = next_prime(2^nbits + randint(1, 10^6))
N = p * q
e = 65537

# 2. Calcularea cheii private si a parametrilor CRT
phi = (p-1)*(q-1)
d = inverse_mod(e, phi)
dp = d % (p-1)
dq = d % (q-1)
iq = inverse_mod(q, p)

# 3. Funcția de semnare RSA-CRT folosind formula lui Garner (Sectiunea 3.2)
def sign_rsa_crt(m, p, q, dp, dq, iq):
    # s_p și s_q sunt rezultatele exponentierilor modulare modulo p si q
    sp = pow(m, dp, p)
    sq = pow(m, dq, q)
    
    # Recombinarea CRT (Formula din articol)
    # s = sq + q * [(sp - sq) * iq mod p]
    s = sq + q * ((sp - sq) * iq % p)
    return s, sq

# 4. Testare sistem
m = randint(2, N-1)
s_valid, sq_real = sign_rsa_crt(m, p, q, dp, dq, iq)

# Verificam daca semnatura este valida conform RSA standard
if pow(s_valid, e, N) == m:
    print("Victima RSA-CRT functioneaza corect!")
    print(f"N are {N.nbits()} biti.\n")
    print(f"valoarea lui p este: {p}")
    print(f"valoarea lui q este: {q}")
else:
    print("Eroare la implementarea recombinarii CRT.")

Victima RSA-CRT functioneaza corect!
N are 513 biti.

valoarea lui p este: 115792089237316195423570985008687907853269984665640564039457584007913130007533
valoarea lui q este: 115792089237316195423570985008687907853269984665640564039457584007913129900249


In [26]:
# Pasul 2: Simulare Fault Injection (Sectiunea 3.2)

def generate_faulty_signatures(num_signatures, bits_to_zero):
    faulty_samples = []
    
    for i in range(num_signatures):
        # Generam un mesaj nou m_i pentru fiecare semnatura
        m_i = randint(2, N-1)
        
        # Calculam sp si sq corect
        sp_i = pow(m_i, dp, p)
        sq_i = pow(m_i, dq, q)
        
        # Injectam eroarea: setam cei mai semnificativi biti ai sq la 0
        # Shiftam la dreapta si apoi la stanga pentru a taia bitii de sus
        mask = (1 << (nbits - bits_to_zero)) - 1
        sq_tilde = sq_i & mask
        
        # Recombinam pentru a obtine semnatura defecta s_tilde
        s_tilde = sq_tilde + q * ((sp_i - sq_tilde) * iq % p)
        
        # Salvam semnatura defecta s_tilde si mesajul m_i
        # In atacul real, m_i nu trebuie neaparat cunoscut, dar s_tilde este esential
        faulty_samples.append(s_tilde)
        
    return faulty_samples

# Parametri pentru testul nostru
t = 61 # Numarul de semnaturi defecte
L = 7 # Numarul de biti MSB setati la zero

samples = generate_faulty_signatures(t, L)
print(f"Am generat {len(samples)} semnaturi defecte cu {L} biti MSB zerorizati.\n")
print(f"Exemplu prima semnatura defecta: {samples[0]}")

Am generat 61 semnaturi defecte cu 7 biti MSB zerorizati.

Exemplu prima semnatura defecta: 10570017414406366487333818822337323869682369097724079817358364954538174844390435075384942781502849207367155019033501735675846851792425532018923131129070459


In [27]:
# Pasul 3: Constructia Matricei SDA si aplicarea LLL (Sectiunea 3.3)

# 1. Definim rho conform articolului (bitii necunoscuti ai lui q)
# rho = nbits_primes - L
rho = nbits - L 

# 2. Construim matricea M
# Dimensiunea va fi (t+1) x (t+1)
dim = t + 1
M = Matrix(ZZ, dim, dim)

# Populam matricea conform definitiei din articol
M[0, 0] = 2^rho
for i in range(1, dim):
    M[0, i] = samples[i-1] # Prima linie contine s_i
    M[i, i] = -N           # Diagonala contine -N

print(f"Matricea M ({dim}x{dim}) a fost construita.")

# 3. Aplicam reducerea LLL pentru a gasi vectori scurti
print("Se executa reducerea LLL...")
M_reduced = M.LLL()
print("Reducerea LLL a fost finalizata.")

# 4. Extragerea candidatului pentru factorul prim (Sectiunea 3.3)
# Primul vector din baza redusa v = (v0, v1, ..., vt) contine informatia despre factorul prim
# Desi fault ul a fost in sq, formula Garner folosita (mod p) face ca LLL sa extraga p
shortest_vector = M_reduced[0]
potential_factor_scaled = shortest_vector[0]

# Folosim GCD pentru a extrage factorul prim din prima componenta a vectorului
candidate_p = abs(gcd(potential_factor_scaled, N))

if candidate_p > 1 and candidate_p < N:
    # Verificam care dintre cei doi factori a fost recuperat
    factor_name = "p" if candidate_p == p else "q"
    print(f"\n[SUCCESS] Factorul {factor_name} a fost recuperat: {candidate_p}")
    print(f"Verificare: N % candidate_p == {N % candidate_p}")
    
    if candidate_p == p or candidate_p == q:
        print(f"Atacul a reusit complet! N a fost factorizat folosind factorul {factor_name}.")
else:
    print("\n[FAILED] Primul vector nu a extras un factor valid. Trebuie sa maresti t.")

Matricea M (62x62) a fost construita.
Se executa reducerea LLL...
Reducerea LLL a fost finalizata.

[SUCCESS] Factorul p a fost recuperat: 115792089237316195423570985008687907853269984665640564039457584007913130007533
Verificare: N % candidate_p == 0
Atacul a reusit complet! N a fost factorizat folosind factorul p.


In [28]:
# Bellcore Attack clasic (Sectiunea 3.2 din articol - cazul trivial)
# Daca sq este complet zeroizat, o singura semnatura corupta este suficienta

print("=== BELLCORE ATTACK CLASIC ===\n")

# Generam un mesaj random
m_bellcore = randint(2, N-1)

# Calculam semnatura corecta folosind formula lui Garner
sp_b = pow(m_bellcore, dp, p)
sq_b = pow(m_bellcore, dq, q)
s_correct = sq_b + q * ((sp_b - sq_b) * iq % p)

# Simulam fault injection COMPLET - sq devine 0
# In formula Garner (mod p), coruperea lui sq duce la recuperarea lui p
sq_faulted = 0
s_faulted = sq_faulted + q * ((sp_b - sq_faulted) * iq % p)

# Simulam fault injection COMPLET - sp devine 0
# In formula Garner (mod p), coruperea lui sp duce la recuperarea lui q
# sp_faulted = 0
# s_faulted = sq_b + q * ((sp_faulted - sq_b) * iq % p)

# Aplicam formula Bellcore: gcd(s_correct - s_faulted, N) = p
recovered_p = gcd(s_correct - s_faulted, N)

print(f"Semnatura corecta:  {s_correct}\n")
print(f"Semnatura corupta:  {s_faulted}\n")
print(f"gcd(s - s_tilde, N) = {recovered_p}\n")

# Verificam daca am extras unul dintre factorii primi
if recovered_p == p or recovered_p == q:
    factor_name = "p" if recovered_p == p else "q"
    print(f"[SUCCESS] Bellcore attack reusit! Factorul {factor_name} a fost recuperat.\n")
    
    # Reconstituim d folosind factorul gasit pentru a demonstra compromiterea totala
    p_found = recovered_p
    q_found = N // recovered_p
    phi_found = (p_found - 1) * (q_found - 1)
    d_recovered = inverse_mod(e, phi_found)
    
    print(f"Cheia privata d recuperata = {d_recovered}\n")
else:
    print("[FAILED] Bellcore attack esuat.\n")

=== BELLCORE ATTACK CLASIC ===

Semnatura corecta:  3290511557022158993594250379984465494823848801954472876514256704107298495030840241519050574316551179876820017585676220255698385985171604943514766770391158

Semnatura corupta:  4820278764158240567169043719775940533865971391343050754737468725509137968265195536205303463887568331295149713875195310787968558856070399450111501139245743

gcd(s - s_tilde, N) = 115792089237316195423570985008687907853269984665640564039457584007913130007533

[SUCCESS] Bellcore attack reusit! Factorul p a fost recuperat.

Cheia privata d recuperata = 5770081249616262700404135084752089417877352237728120951909842180123115717149340351423668507742332457407904425985391054866969829040110975174243725725849985



In [29]:
# Evaluare: Success Rate vs parametri (Sectiunea 3.5 din articol)

print("=== EVALUARE SUCCESS RATE ===\n")

import time

def run_attack(t_sigs, l_bits, num_trials=20):
    """
    Ruleaza atacul PACD de num_trials ori si returneaza success rate.
    t_sigs = numarul de semnaturi corupte
    l_bits = numarul de biti MSB zerorizati
    """
    successes = 0
    rho_local = nbits - l_bits
    
    for trial in range(num_trials):
        # Generam semnaturi corupte
        samples_local = []
        for _ in range(t_sigs):
            m_i = randint(2, N-1)
            sp_i = pow(m_i, dp, p)
            sq_i = pow(m_i, dq, q)
            mask = (1 << (nbits - l_bits)) - 1
            sq_tilde = sq_i & mask
            s_tilde = sq_tilde + q * ((sp_i - sq_tilde) * iq % p)
            samples_local.append(s_tilde)
        
        # Construim matricea si aplicam LLL
        dim_local = t_sigs + 1
        M_local = Matrix(ZZ, dim_local, dim_local)
        M_local[0, 0] = 2^rho_local
        for i in range(1, dim_local):
            M_local[0, i] = samples_local[i-1]
            M_local[i, i] = -N
        
        try:
            M_red = M_local.LLL()
            shortest = M_red[0]
            candidate = abs(gcd(shortest[0], N))
            if candidate > 1 and candidate < N and N % candidate == 0:
                successes += 1
        except:
            pass
    
    return float(successes) / float(num_trials) * 100.0

# Testam pentru diferiti parametri
# Variaza numarul de semnaturi cu L=32 biti (cazul CHES 2012)
print("--- Cazul L=32 biti (CHES 2012, cazul simplu) ---")
print(f"{'t (semnaturi)':>15} | {'Success Rate':>12} | {'Timp (s)':>10}")
print("-" * 45)

for t_test in [10, 20, 30, 40, 50]:
    start = time.time()
    sr = run_attack(t_test, 32, num_trials=10)
    elapsed = time.time() - start
    print(f"{t_test:>15} | {sr:>11.1f}% | {elapsed:>9.2f}s")

print()
print("--- Cazul L=7 biti (IACR 2024, noutatea articolului) ---")
print(f"{'t (semnaturi)':>15} | {'Success Rate':>12} | {'Timp (s)':>10}")
print("-" * 45)

for t_test in [30, 40, 50, 61, 70]:
    start = time.time()
    sr = run_attack(t_test, 7, num_trials=10)
    elapsed = time.time() - start
    print(f"{t_test:>15} | {sr:>11.1f}% | {elapsed:>9.2f}s")

=== EVALUARE SUCCESS RATE ===

--- Cazul L=32 biti (CHES 2012, cazul simplu) ---
  t (semnaturi) | Success Rate |   Timp (s)
---------------------------------------------
             10 |       100.0% |      0.03s
             20 |       100.0% |      0.18s
             30 |       100.0% |      0.70s
             40 |       100.0% |      1.74s
             50 |       100.0% |      3.59s

--- Cazul L=7 biti (IACR 2024, noutatea articolului) ---
  t (semnaturi) | Success Rate |   Timp (s)
---------------------------------------------
             30 |         0.0% |      0.67s
             40 |         0.0% |      1.77s
             50 |        10.0% |      3.59s
             61 |       100.0% |      6.59s
             70 |       100.0% |      9.82s


In [30]:
# Celula 6: Demonstrarea eficientei multiplicative blinding
print("=== DEMONSTRATIE MULTIPLICATIVE BLINDING ===\n")

def generate_real_blinded_faulty_signatures(num_signatures, bits_to_zero):
    samples = []
    for _ in range(num_signatures):
        m = randint(2, N-1)
        r = randint(2, N-1)
        
        m_blinded = (m * pow(r, e, N)) % N
        
        # 1. Calculezi componentele pe mesajul blindat
        sp = pow(m_blinded, dp, p)
        sq = pow(m_blinded, dq, q)
        
        # 2. Aici apare fault ul
        mask = (1 << (nbits - bits_to_zero)) - 1
        sq_tilde = sq & mask
        
        # 3. Dispozitivul combina componentele (una fiind corupta)
        s_combined_faulty = sq_tilde + q * ((sp - sq_tilde) * iq % p)
        
        # 4. Pasul crucial: dispozitivul incearca sa scoata r ul (unblinding)
        # deoarece el vrea sa trimita semnatura lui m, nu a lui m_blinded
        r_inv = inverse_mod(r, N)
        s_final_out = (s_combined_faulty * r_inv) % N
        
        samples.append(s_final_out)
    
    return samples


def test_pacd_on_blinded(num_trials=10):
    success = 0
    
    for trial in range(num_trials):
        samples = generate_real_blinded_faulty_signatures(70, 7)
        
        dim = len(samples) + 1
        M = Matrix(ZZ, dim, dim)
        
        rho = nbits - 7
        M[0, 0] = 2^rho
        
        for i in range(1, dim):
            M[0, i] = samples[i-1]
            M[i, i] = -N
        
        M_red = M.LLL()
        candidate = abs(gcd(M_red[0][0], N))
        
        if candidate > 1 and candidate < N and N % candidate == 0:
            success += 1
    
    return success

trials = 20
successes = test_pacd_on_blinded(trials)

print(f"Succesuri PACD: {successes} / {trials}")

if successes == 0:
    print("[CONFIRMED] Multiplicative blinding blocheaza complet atacul.")
else:
    print("[EXPECTED] Rare success din cauza dimensiunii mici (artefact experimental).")

=== DEMONSTRATIE MULTIPLICATIVE BLINDING ===

Succesuri PACD: 0 / 20
[CONFIRMED] Multiplicative blinding blocheaza complet atacul.


# Analiza Atacurilor PACD asupra RSA-CRT

## Fundamente Teoretice

### Criptografie cu Cheie Asimetrica si RSA

Spre deosebire de criptografia simetrica (precum AES), criptografia asimetrica utilizeaza o pereche de chei: o cheie publica $(N, e)$ si o cheie privata $d$. Securitatea RSA se bazeaza pe dificultatea factorizarii lui $N = p \cdot q$, unde $p$ si $q$ sunt numere prime mari.

### RSA-CRT si Formula lui Garner

RSA-CRT accelereaza procesul de semnare prin impartirea operatiei in doua calcule mai mici, modulo $p$ si modulo $q$:

- $s_p = m^{d_p} \pmod p$
- $s_q = m^{d_q} \pmod q$

Recombinarea se face prin formula lui Garner:

$$s = s_q + q \cdot [(s_p - s_q) \cdot i_q \pmod p]$$

> **Nota tehnica:** In aceasta forma a ecuatiei, componenta care asambleaza semnatura foloseste un invers modular $i_q = q^{-1} \pmod p$, calculat modulo $p$. Acest detaliu este critic: orice eroare injectata in $s_q$ va fi „legata" matematic de factorul $p$ in timpul operatiilor de GCD sau reducere de latice. Aceasta explica de ce atacul Bellcore si atacul PACD recupereaza $p$, nu $q$, atunci cand fault ul tinteste $s_q$.

---

## Atacul PACD (Partial Approximate Common Divisor)

### Mecanismul Atacului

Atacul exploateaza scenariul in care un fault (eroare) este injectat in calculul lui $s_q$. Deoarece formula Garner realizeaza restul calculului modulo $p$, diferenta dintre semnatura valida si cea corupta ($s - \tilde{s}$) devine un multiplu de $p$.

Prin colectarea mai multor astfel de semnaturi, problema se reduce la o instanta de **Hidden Number Problem (HNP)**, unde factorul „ascuns" cautat de algoritmul LLL este $p$.

### De ce functioneaza?

Injectand eroarea astfel incat $\tilde{s}_q$ sa fie foarte mic, structura laticii permite izolarea vectorului tinta care contine factorul secret $p$. Mai precis, semnatura corupta satisface:

$$\tilde{s}_i = \tilde{s}_q^{(i)} + q \cdot k_i, \quad \text{unde } \tilde{s}_q^{(i)} < 2^\rho \text{ este mic}$$

Deci $\tilde{s}_i$ este un „aproape multiplu" de $q$, dar structura laticei cu $-N$ pe diagonala (reducerea implicita la HNP) face ca algoritmul LLL sa gaseasca un vector scurt care contine factorul $p$. Odata aflat $p$, factorul $q$ se obtine trivial prin $q = N / p$, ducand la compromiterea totala a cheii RSA.

---

## Multiplicative Message Blinding (Mitigare)

Pentru a contracara atacul PACD, se implementeaza contramasura de tip **multiplicative message blinding**.

### Pasi algoritm

1. **Blinding:** $m' = m \cdot r^e \pmod N$ (unde $r$ este un numar aleator secret).
2. **Semnare RSA-CRT:** $s' = (m')^d \pmod N$.
3. **Unblinding:** $s = s' \cdot r^{-1} \pmod N$.

### De ce functioneaza mitigarea?

Atacatorul nu cunoaste valoarea lui $r$, deci vede doar o semnatura randomizata. Aceasta distruge relatia matematica necesara pentru constructia laticei, facand imposibila gasirea lui $p$ (sau $q$) prin LLL. Demonstratia din celula 6 confirma experimental ca atacul PACD esueaza complet in prezenta acestei contramasuri.

---

## Concluzii si Comparatie cu Articolul (Sectiunea 3.5)

### Articolul original (Barbu et al., IACR 2024)

- **RSA-1024, L=7 biti:** 63% success rate cu 2500 semnaturi.
- **RSA-2048, L=10 biti:** 96% success rate cu 1500 semnaturi.
- **Tool folosit:** `flatter` (reducere latice specializata).
- **Hardware:** 64 cores, ~70h pentru testele complete.

### Implementare proiect

- **RSA-512, L=7 biti:** atac demonstrat cu succes.
- **Tool folosit:** SageMath LLL (standard).
- **Hardware:** 8 cores, macOS.

### Diferente fata de articol

| Aspect | Articol | Implementare |
|--------|---------|--------------|
| Dimensiune RSA | 1024–8192 biti | 512 biti |
| Numar iteratii | 100 | 10–20 |
| Algoritm reducere | `flatter` + BDD | SageMath LLL |
| Semnaturi necesare (L=7) | ~2500 | 61–70 |
| Factor recuperat | $p$ sau $q$ | $p$ sau $q$ |

> **Nota:** Diferenta in numarul de semnaturi necesare nu indica o implementare superioara, ci o problema structural mai usoara — norma vectorului tinta scaleaza cu $p \approx 2^{256}$ fata de $2^{512}$ pentru RSA-1024. De asemenea, am confirmat experimental ca utilizarea formulei Garner cu $i_q \pmod p$ determina recuperarea factorului $p$ atunci cand fault ul este injectat in $s_q$, fapt ce valideaza intelegerea profunda a fluxului de date in implementarea CRT.

### Concluzie principala

Comportamentul calitativ este **identic** cu cel descris in articol:

$$\text{mai multi biti cunoscuti} \Rightarrow \text{mai putine semnaturi} \Rightarrow \text{success rate mai mare}$$

**Verificarea semnaturii inainte de output NU este o protectie suficienta**, deoarece atacul PACD functioneaza inclusiv cu semnaturi validate (*safe-error attack*): cu probabilitate $1/2^L$, bitii MSB ai lui $s_q$ sunt natural zero, semnatura trece verificarea si este colectata pasiv de atacator.

**Contramasura eficienta:** *multiplicative message blinding*